<div align="center">
  <img src="https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/wsu_logo_horizontal.png" alt="Wayne State University Logo" width="320">
  <h1>Lab 4: Regression for Tool Wear Estimation and Prediction</h1>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WSU-AI-in-ME/ai-in-me-1/blob/main/labs/week04/lab04_tool_wear_regression.ipynb)

[Course Repository](https://github.com/WSU-AI-in-ME/ai-in-me-1) · [Lab Index](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/Lab_index.ipynb)

### ME 5995 — AI in Mechanical Engineering I: Fundamentals of Manufacturing Data Science
**Wayne State University**

**Graded individual Lab · Approximately 90 minutes**
**Version:** Student Notebook

Open in Colab and save a personal copy before editing. Canvas contains the official submission requirements and due date.

## Student Information

Double-click this Markdown cell to edit.

**Student name:** TODO: Enter your name  
**WSU AccessID:** TODO: Enter your WSU AccessID

## The complete workflow: measurements to wear estimation

![Instructor concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/experiment_overview.png)

**Read the workflow from left to right:** cutting produces force, vibration,
and AE-RMS recordings. Wear measurements provide the reference target. The course
feature table summarizes each cut in one row; regression uses those sensor
summaries to estimate mean flute wear. In this Lab, start with the prepared CSV:
you do not need to collect raw signals or repeat feature extraction.

**Read the equation:** the small plot is a single-feature illustration,
not a fitted result for this dataset. The prediction and observed value are related by

$$\hat y_i=\beta_0+\beta_1x_i,\qquad
e_i=y_i-\hat y_i,\qquad y_i=\hat y_i+e_i.$$

The actual Lab uses the same **11 sensor features** for every model. Cut number
orders the records and labels plots; it is not a predictor here. The recorded
acoustic channel is AE RMS. In the setup annotation, the feed unit is mm/min.


## Overview: build first, then judge where the model works

Can force, vibration and AE-RMS summaries estimate a tool's wear? We use
measured wear as the target and compare four regression candidates using
**exactly the same 11 sensor inputs**. A model may fit early cuts well and
still struggle on later cuts. Both outcomes are useful evidence.

**Workflow:** split → prepare X/y → scale → fit → predict → validate/select
→ refit → final test → engineering interpretation.

By the end, you can:
- identify predictors, a continuous target and their units;
- follow a complete linear-regression example and apply the same steps to
  Ridge, second-degree Polynomial regression and RBF SVR;
- compare three Ridge alpha and three SVR C values using validation RMSE;
- interpret final error in micrometers and state the evaluation's limits.

**Your work:** four short checkpoint responses (Checkpoint 2 includes a hypothesis before running) and four small fit/predict edits.
Run supplied cells unchanged. You do not need to write plotting functions,
loops, feature selection or parameter-search code. An error-free starter
is not a completed submission: unfinished models print reminders and skip
dependent results until you complete the edits.

**Time guide:** guided preparation/example 40 min; similar model work 30 min;
selection, final evaluation and submission 20 min. Total: **90 min**, estimated.
No separate report. Canvas controls the due date and point allocation.

**Reading the code:** focus on the model creation, `fit`, and `predict` lines. Cells marked **PROVIDED — RUN UNCHANGED** handle repetition, tables, and figures. Read the short explanation above each block, run it, then inspect its output. You do not need to reproduce the supplied infrastructure.

| Reading route | What to focus on |
|---|---|
| **Understand and complete** | The data split; X/y and scaling; the Linear fit/predict example; four TODO expressions; four checkpoint responses. |
| **Run and interpret** | The supplied parameter experiments, metric tables, selection and final plots. Understand their purpose and results; you do not need to reproduce their loops or plotting syntax. |
| **Optional** | Full-c6 inspection and the additional interpretation challenge; neither adds required written work. |

Use the detailed line comments as a reference when a step is unclear. You do
not need to memorize all supplied Python code. The 90-minute estimate includes
reading and interpretation; it is not a measured beginner completion time.


**First pass:** run cells in order, write the CP2 hypothesis before its experiments,
and record your CP3 choice before final test evaluation. **Submission check:**
after completing the responses and code, restart and Run all to reproduce the
saved results. Reproducing a locked analysis is different from retuning on test results.

**Scoring: 10 points.** CP1: 2; CP2: 4 (implementation 2, experiment/interpretation 2);
CP3: 2; CP4: 2. Optional work is ungraded. Scores reward completion and reasonable
evidence-based effort, not achieving a particular prediction accuracy.


## Software, files and dataset context

Use NumPy, pandas, Matplotlib and scikit-learn. Colab users can run imports
directly. Locally, install these packages and Jupyter in your local Python
environment. If a requirements.txt file is provided, you may use it to install
those packages. Put `phm2010_features.csv` beside this notebook for offline use,
or retain the course repository directory structure. Internet access allows
the supplied loader to retrieve the course CSV.

The original **PHM Society 2010 Data Challenge** includes cutter recordings
c1–c6; wear labels are available for c1, c4 and c6. This course-derived table
has one row per cut: 315 cuts from each labeled cutter, 945 rows total.
The primary target `wear_mean_um` is the arithmetic mean of the three
recorded flute-wear values, in µm. It is a course-derived target.

The seven signal channels comprise three forces (N), three vibrations (g)
and one AE-RMS channel (V), documented at 50 kHz. Features summarize full
recordings without an inferred active-cut segment; raw signals are not
needed here. The 11 defaults were chosen for physical interpretation and
redundancy considerations, not optimized for this Lab's test score.

This is a **course-specific estimation task**, not a reproduction of the
original competition. Three cutters are three physical groups, not 945
independent experiments. Repository MIT licensing does not override PHM
dataset rights; see the feature README for the unresolved original-license
status and third-party mirror distinction. Do not infer redistribution rights.

[Feature README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md) ·
[Data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md)

In [ ]:
# [PROVIDED SETUP] Import tools; importing does not fit a model.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Readable labels are used in all supplied figures.
plt.rcParams.update({'font.size': 11, 'figure.dpi': 110})

In [ ]:
# [PROVIDED SETUP] Prefer a local CSV; otherwise read the existing course file.
# Path represents a file location. No source data are modified.
data_path = Path('phm2010_features.csv')
if not data_path.exists():
    data_path = Path('../../data/phm2010/features/phm2010_features.csv')
if data_path.exists():
    features = pd.read_csv(data_path)
else:
    features = pd.read_csv('https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/data/phm2010/features/phm2010_features.csv')

# Each table below contains one cutter, ordered from earlier to later cuts.
c1 = features.loc[features['cutter_id'] == 'c1'].sort_values('cut_number')
c4 = features.loc[features['cutter_id'] == 'c4'].sort_values('cut_number')
c6 = features.loc[features['cutter_id'] == 'c6'].sort_values('cut_number')
print('Rows for c1, c4, c6:', len(c1), len(c4), len(c6))
display(c1[['cutter_id', 'cut_number', 'wear_mean_um']].head())

## Guided preparation: three separate jobs for the data

| Partition | Cuts within EACH cutter | Purpose |
|---|---|---|
| Train, approximately 50% | 1–157 | Learn model coefficients/relationships and scaling |
| Validation, approximately 20% | 158–220 | Select a model and its settings |
| Test, approximately 30% | 221–315 | Evaluate after the choice is fixed |

315 cannot be divided into exact integer percentages. We use boundaries
`int(0.50 * 315) = 157` and `int(0.70 * 315) = 220`.
We combine the three training portions, then the three validation portions,
then the three test portions. We **do not** split the whole stacked table at
one row boundary, because that would split by cutter rather than within cutter.

`.iloc[start:stop]` selects row positions; stop is excluded. Thus `:157`
selects the first 157 rows, and `157:220` selects the next 63.
Chronological splitting matches this later-cut evaluation question; it is
not a universal rule for all datasets. Random mixing asks a different question.

![Instructor concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/training_workflow.png)

Workflow overview. Shading boundaries are between cuts 157/158 and 220/221. The same split is applied within each cutter and matching partitions are pooled. Select using **validation RMSE**. Whole-trajectory curves provide context; do not use the test trajectory to choose settings.

In [ ]:
# [PROVIDED] Keep earlier, middle and later cuts separate within each cutter.
# pd.concat stacks tables vertically; ignore_index gives new row labels.
train = pd.concat([c1.iloc[:157], c4.iloc[:157], c6.iloc[:157]], ignore_index=True)
valid = pd.concat([c1.iloc[157:220], c4.iloc[157:220], c6.iloc[157:220]], ignore_index=True)
test = pd.concat([c1.iloc[220:], c4.iloc[220:], c6.iloc[220:]], ignore_index=True)
print('Train / validation / test rows:', len(train), len(valid), len(test))

# These checks detect an unexpected source table; they are not student tasks.
assert len(train) == 471 and len(valid) == 189 and len(test) == 285
assert train['cut_number'].max() == 157
assert valid['cut_number'].min() == 158 and valid['cut_number'].max() == 220
assert test['cut_number'].min() == 221

# The figure shows partition boundaries only, not held-out target values.
fig, ax = plt.subplots(figsize=(9, 3.3))
ax.barh(['c1', 'c4', 'c6'], [157, 157, 157], left=0.5, color='#086759', label='Train')
ax.barh(['c1', 'c4', 'c6'], [63, 63, 63], left=157.5, color='#C08923', label='Validation')
ax.barh(['c1', 'c4', 'c6'], [95, 95, 95], left=220.5, color='#2468A0', label='Test')
ax.set(xlabel='Cut number (sequence)', ylabel='Cutter', title='Same chronological split within each cutter')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.35), ncol=3, frameon=False)
fig.tight_layout()
plt.show()

## Guided preparation: X is information; y is the answer to estimate

**One row = one cut.** `X_train` has 471 rows and 11 columns. `y_train` has
471 corresponding wear values. Row order matters: every X row must match its y.

`table[list_of_columns]` returns a 2D table. `table['one_column']` returns a
1D series. This is why X uses a list and y uses one column name.
Cutter ID, cut number, the other wear columns and wear classes are not predictors.
Metadata are retained only to split records and interpret grouped results.

### Notation: a number, a vector, and a table

| Symbol | Meaning |
|---|---|
| $x_{ij}$ | One scalar: feature $j$ for cut $i$ |
| $\mathbf{x}_i$ | The 11 features of one cut, a column vector |
| $\mathbf{X}$ | All cuts in a matrix: rows = cuts, columns = features |
| $y_i$, $\hat y_i$ | One measured wear and one predicted wear, in µm |
| $e_i=y_i-\hat y_i$ | One residual; positive means underprediction |
| $\mathbf y$, $\hat{\mathbf y}$, $\mathbf e$ | Vectors of measured wear, predictions, and residuals |

In math, vectors use bold lowercase and matrices use bold uppercase.
`X_train` is a 471 × 11 table; `y_train` is stored as a 1D sequence with
shape `(471,)` in Python. Scaled inputs are $z_{ij}$ and $\mathbf z_i$.
A single-feature illustration uses scalar $x_i$; the real Lab uses 11 inputs.

![Instructor concept diagram](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/sensor_inputs_target.png)

Instructor diagram. Here $\mathbf{x}_i$ is the vector of 11 inputs for one cut; $y_i$, $\hat y_i$, and $e_i$ are scalars. Read the reference as **measured $y_i$: mean of three flute-wear values**. Standardize inputs using training statistics before fitting the model; wear stays in µm.

In [ ]:
# [PROVIDED] Fixed sensor inputs; do not add/remove columns for this Lab.
default_features = [
    'force_x_mean', 'force_x_sd', 'force_y_mean', 'force_y_sd',
    'force_z_mean', 'force_z_sd', 'vibration_x_sd', 'vibration_y_sd',
    'vibration_z_sd', 'ae_rms_mean', 'ae_rms_sd',
]
# Select the 11 sensor columns for training; keep all rows and their order.
X_train = train[default_features]   # Sensor features: forces N, vibrations g, AE-RMS V.
# Use exactly the same columns for validation so each feature keeps its meaning.
X_valid = valid[default_features]   # Same columns, in the same order.
# Select the measured training target as a 1D series; one value per training cut.
y_train = train['wear_mean_um']     # Measured mean flute wear, in micrometers.
# Keep measured validation wear for scoring later; it is not an input to predict.
y_valid = valid['wear_mean_um']
print('Training X shape:', X_train.shape)
print('Training y shape:', y_train.shape)
print('Training wear range (µm):', round(y_train.min(), 2), 'to', round(y_train.max(), 2))
# Test X and y are deliberately not used during candidate selection.

## Required Checkpoint 1 — Describe the regression task

State what one row, X and y represent, with the target unit. Record pooled
train/validation/test counts and explain why we keep cut order within each cutter.

**Your response:** TODO: Write 2–3 sentences.

## Guided preparation: scaling without changing the target unit

A force value in N and a vibration value in g have different numerical scales.
Standardization puts each predictor on a comparable scale:

$$z_{ij} = \frac{x_{ij}-\mu_{j,\mathrm{train}}}{s_{j,\mathrm{train}}}.$$

The scaler learns each column's mean and population standard deviation from
training only. Standardized X is dimensionless. **y stays in µm.**
Scaling does not remove columns or make correlated columns independent.

- `fit`: learn quantities from the supplied data.
- `transform`: apply quantities already learned.
- `fit_transform`: perform both, used on training only.

We transform validation with the training scaler. Fitting a new scaler to
validation would change the coordinate system seen by the trained model.

In [ ]:
# [GUIDED — RUN UNCHANGED] Fit one scaler for candidate comparison.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
print('Scaled training shape:', X_train_scaled.shape)
# A scaled value of 2 means two training standard deviations above the mean.

## Guided example: a complete linear-regression workflow

A single-feature illustration is $\hat y_i=\beta_0+\beta_1x_i$.
With our 11 standardized sensor inputs:

$$\hat y_i=\beta_0+\sum_{j=1}^{11}\beta_j z_{ij}
=\beta_0+\boldsymbol\beta^{\mathsf T}\mathbf z_i.$$

The coefficients and intercept are learned by `fit`; `predict` uses
them with new inputs and does not receive the validation answers.
The hat means predicted. There is no residual term added to the prediction;
the observed value satisfies $y_i=\hat y_i+e_i$ by definition.
Coefficient size alone is not feature importance when inputs are correlated.

In [ ]:
# [GUIDED EXAMPLE] Read this pattern before the two student model cells.
linear_model = LinearRegression()  # Create an unfitted model with an intercept.
linear_model.fit(X_train_scaled, y_train)  # Learn from training inputs AND targets.
linear_valid_pred = linear_model.predict(X_valid_scaled)  # Predict without y_valid.
linear_train_pred = linear_model.predict(X_train_scaled)  # Check fit on known rows.

# Store each quantity explicitly; RMSE is the square root of mean squared error.
linear_valid_mae = mean_absolute_error(y_valid, linear_valid_pred)
linear_valid_rmse = np.sqrt(mean_squared_error(y_valid, linear_valid_pred))
linear_valid_r2 = r2_score(y_valid, linear_valid_pred)
linear_train_rmse = np.sqrt(mean_squared_error(y_train, linear_train_pred))
print('Training RMSE (µm):', round(linear_train_rmse, 2))
print('Validation MAE (µm):', round(linear_valid_mae, 2))
print('Validation RMSE (µm):', round(linear_valid_rmse, 2))
print('Validation R² (unitless):', round(linear_valid_r2, 3))

### How to read the numbers and residuals

For measured wear $y_i$, prediction $\hat y_i$, and $n$ evaluated cuts:

$$e_i=y_i-\hat y_i,\quad
\mathrm{MAE}=\frac{1}{n}\sum_i|e_i|,\quad
\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_i e_i^2}.$$

MAE is mean absolute error; RMSE gives large errors more weight. Both are in
**µm**, and smaller is better. A residual above zero means underestimation.

$$R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y)^2}.$$

Here $\bar y$ is the mean of the **evaluated partition**. R² is unitless,
equals 1 for perfect predictions, and can be negative. A negative value means
larger squared error than predicting that partition's mean. That reference
mean is not a deployable prediction learned from training.

The supplied baseline below instead predicts the **training mean** everywhere.
It provides a simple reference, not a fifth candidate in this exercise.

In [ ]:
# [PROVIDED] A baseline learned from training only.
baseline_valid_pred = np.full(len(y_valid), y_train.mean())
baseline_valid_rmse = np.sqrt(mean_squared_error(y_valid, baseline_valid_pred))
print('Training-mean baseline validation RMSE (µm):', round(baseline_valid_rmse, 2))

# [PROVIDED PLOT] Display one cutter to explain the visual before the comparison.
c1_valid_mask = valid['cutter_id'] == 'c1'
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(valid.loc[c1_valid_mask, 'cut_number'], y_valid[c1_valid_mask], label='Measured')
axes[0].plot(valid.loc[c1_valid_mask, 'cut_number'], linear_valid_pred[c1_valid_mask], label='Predicted')
axes[0].set(title='Linear regression: c1 validation', xlabel='Cut number', ylabel='Mean flute wear (µm)')
axes[0].legend()
axes[1].plot(valid.loc[c1_valid_mask, 'cut_number'], y_valid[c1_valid_mask] - linear_valid_pred[c1_valid_mask])
axes[1].axhline(0, color='black', linestyle='--')
axes[1].set(title='Measured minus predicted', xlabel='Cut number', ylabel='Residual (µm)')
fig.tight_layout()
plt.show()

## Required Checkpoint 2 — Build models and vary one setting

Follow the Linear example: complete one `fit` and one `predict` call
for Ridge, then for SVR (**four code edits total**). Polynomial is provided.
`None` means unfinished; later results wait for your predictions.

### Ridge: keep a linear model, penalize large coefficients

$$\min_{\beta_0,\boldsymbol\beta}
\underbrace{\sum_{i\in\mathrm{train}}(y_i-\hat y_i)^2}_{\text{fit the training data}}
+\underbrace{\alpha\sum_{j=1}^{11}\beta_j^2}_{\text{penalize large coefficients}}.$$

Larger $\alpha$ puts more weight on coefficient shrinkage. The intercept
is not penalized. This does not guarantee a lower validation error and
is not feature selection. We compare **alpha = 0.1, 1, 10** on the same split.
First complete the model below at alpha = 1; supplied code repeats the
same process for the three values. No loops need to be written.

**Required parameter experiment:** keep the initial examples at alpha = 1 and
C = 100. Complete the four fit/predict expressions, then run the supplied loops
to compare the three listed values for each parameter. Predict a trend first,
inspect the curves, and explain your observation. Selection uses those lists,
not a manually edited starting value.

**Optional exploration, before final test only:** try one of the supplied values
in an initial model cell and inspect its validation predictions. This does not
change the experiment's candidate lists. Restore the initial value and rerun
the experiments and selection in order before continuing. Do not retune after
viewing the test results.


After each task cell, inspect its **initial validation RMSE**. Those predictions
are preserved separately from the supplied tuning results. Successful final
plots do not prove that the four task expressions are correct: check which
inputs and targets you passed, including their row order. In optional exploration,
restore alpha=1 and C=100 and rerun the task/feedback cells before comparison.


![Illustrative synthetic data, not PHM results. Ridge remains a straight line for one input; degree-2 Polynomial can curve. Greater flexibility need not improve later-cut accuracy.](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/regression_families.png)

Illustrative synthetic data, not PHM results. Ridge remains a straight line for one input; degree-2 Polynomial can curve. Greater flexibility need not improve later-cut accuracy.

The parameter values in this synthetic illustration were chosen to make the concept visible; use the specified PHM candidate lists for the required Lab experiment.

In [ ]:
# [STUDENT TASK] Complete the two TODO expressions; keep other lines unchanged.
# Step 1 — Create the model object that will remember the fitted relationship.
# alpha controls the penalty on large coefficients; 1.0 is our starting value.
# Creating this object does not learn coefficients yet.
ridge_model = Ridge(alpha=1.0)

# Step 2 — Learn from the training data, following the completed Linear example.
# X_train_scaled: 471 cuts × 11 standardized sensor features (unitless).
# y_train: the 471 matching measured wear values (µm), in the same row order.
# Replace the TODO comment with a fit call on the model created above.
# Fitting updates that model object; do not use validation or test targets here.
# TODO 1: Fit ridge_model using X_train_scaled and y_train.

# Step 3 — Apply the fitted relationship to the validation inputs.
# X_valid_scaled: 189 cuts × the same 11 features, scaled with TRAINING statistics.
# Replace None with a prediction call on your fitted model. Do not pass y_valid.
# The result is a 1D array of 189 predicted wear values (µm), one per input row.
# The equals sign stores these predictions under the variable name on the left.
ridge_valid_pred = None  # TODO 2: Predict using X_valid_scaled.
# None is only an unfinished placeholder; downstream cells wait until it is replaced.

In [ ]:
# [PROVIDED FEEDBACK] Keep the initial student result separate from later tuning.
ridge_initial_model = ridge_model
ridge_initial_pred = ridge_valid_pred
if ridge_initial_pred is not None:
    # A valid prediction has one finite numeric value for each validation cut.
    ridge_initial_pred = np.asarray(ridge_initial_pred)
    assert ridge_initial_pred.shape == (len(y_valid),), 'Expected one prediction per validation cut.'
    assert np.isfinite(ridge_initial_pred).all(), 'Predictions must contain finite numbers.'
    ridge_initial_rmse = np.sqrt(mean_squared_error(y_valid, ridge_initial_pred))
    print('Initial Ridge validation RMSE (µm):', round(ridge_initial_rmse, 3))
    print('This is your initial model, before the supplied parameter comparison.')
else:
    print('Complete the two Ridge expressions above to see initial feedback.')


### Candidate 3: second-degree Polynomial regression

A two-input illustration is $\hat y_i=\beta_0+\beta_1z_{i1}+\beta_2z_{i2}+\beta_3z_{i1}^2+\beta_4z_{i1}z_{i2}+\beta_5z_{i2}^2$.
Squares allow curvature; a product allows one input's effect to depend on
another input. The equation is nonlinear in the original inputs but linear
in its coefficients, so we still use `LinearRegression` after expansion.

Our 11 inputs become **77 columns**: 11 originals, 11 squares and 55 pairwise
products. More flexibility does not guarantee better later-cut predictions.
`include_bias=False` avoids a duplicate constant because the regression
already has an intercept. Run the transformation code unchanged.

In [ ]:
# [PROVIDED] Expand the training and validation columns in exactly the same way.
polynomial = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = polynomial.fit_transform(X_train_scaled)
X_valid_poly = polynomial.transform(X_valid_scaled)
print('Polynomial columns:', X_train_poly.shape[1])

In [ ]:
# [PROVIDED COMPARISON] Degree stays fixed; this is not a student code task.
polynomial_model = LinearRegression()
polynomial_model.fit(X_train_poly, y_train)
polynomial_valid_pred = polynomial_model.predict(X_valid_poly)

### Support Vector Regression (SVR): an epsilon-insensitive loss

RBF SVR uses similarity between scaled input rows to represent a nonlinear
relation. Use the original **11 scaled columns**, not polynomial features.

$$L_{\epsilon_{\mathrm{SVR}}}(e_i)
=\max(0,\,|e_i|-\epsilon_{\mathrm{SVR}}).$$

| Setting | Meaning / our choice |
|---|---|
| `epsilon=1.0` | Zero loss within ±1 µm of the prediction; fixed here. This is not a confidence interval or guaranteed accuracy. |
| `C` | Weight on loss outside the band relative to model complexity; compare **10, 100, 1000**. |
| `gamma='scale'` | Training-data rule for RBF similarity width; keep this rule fixed. |

Larger C penalizes violations more strongly, but does not guarantee better
validation performance. $e_i$ is the residual; $\epsilon_{\mathrm{SVR}}$
is a setting. Target wear remains in µm. Complete the example at C = 100.

![Illustrative synthetic data with epsilon = 1 µm. The shaded band visualizes zero-loss residuals; only the distance outside it is penalized.](https://raw.githubusercontent.com/WSU-AI-in-ME/ai-in-me-1/main/img/lab04/svr_band.png)

Illustrative synthetic data with epsilon = 1 µm. The shaded band visualizes zero-loss residuals; only the distance outside it is penalized.

The parameter values in this synthetic illustration were chosen to make the concept visible; use the specified PHM candidate lists for the required Lab experiment.

In [ ]:
# [STUDENT TASK] Complete the two TODO expressions; keep other lines unchanged.
# Step 1 — Create the model object that will remember the fitted relationship.
# kernel='rbf' allows a nonlinear relation based on similarity between input rows.
# C=100.0 is the starting penalty weight; epsilon=1.0 sets a 1 µm zero-loss band.
# gamma='scale' uses a training-input rule for the similarity width.
# These settings define the model; creating it does not fit it yet.
svr_model = SVR(kernel='rbf', C=100.0, epsilon=1.0, gamma='scale')

# Step 2 — Learn from the training data, following the completed Linear example.
# X_train_scaled: 471 cuts × 11 standardized sensor features (unitless).
# y_train: the 471 matching measured wear values (µm), in the same row order.
# Replace the TODO comment with a fit call on the model created above.
# Fitting updates that model object; do not use validation or test targets here.
# TODO 3: Fit svr_model using X_train_scaled and y_train.

# Step 3 — Apply the fitted relationship to the validation inputs.
# X_valid_scaled: 189 cuts × the same 11 features, scaled with TRAINING statistics.
# Replace None with a prediction call on your fitted model. Do not pass y_valid.
# The result is a 1D array of 189 predicted wear values (µm), one per input row.
# The equals sign stores these predictions under the variable name on the left.
svr_valid_pred = None  # TODO 4: Predict using X_valid_scaled.
# None is only an unfinished placeholder; downstream cells wait until it is replaced.

In [ ]:
# [PROVIDED FEEDBACK] Keep the initial student result separate from later tuning.
svr_initial_model = svr_model
svr_initial_pred = svr_valid_pred
if svr_initial_pred is not None:
    # A valid prediction has one finite numeric value for each validation cut.
    svr_initial_pred = np.asarray(svr_initial_pred)
    assert svr_initial_pred.shape == (len(y_valid),), 'Expected one prediction per validation cut.'
    assert np.isfinite(svr_initial_pred).all(), 'Predictions must contain finite numbers.'
    svr_initial_rmse = np.sqrt(mean_squared_error(y_valid, svr_initial_pred))
    print('Initial Svr validation RMSE (µm):', round(svr_initial_rmse, 3))
    print('This is your initial model, before the supplied parameter comparison.')
else:
    print('Complete the two Svr expressions above to see initial feedback.')


### Checkpoint 2 response — before running the experiment

Predict how increasing alpha or C might change training and validation
RMSE. One sentence is enough; a reasonable hypothesis need not be correct.
After running Experiments A/B and the tables/plots below, return here and
add one sentence describing what you observed.

**Your response:** TODO: Hypothesis first; then one observation.

### Experiment A — repeat Ridge for three alpha values

**Run unchanged.** One pass through `for` fits one candidate. `alpha` takes 0.1, then 1, then 10. Each result row stores the setting, training RMSE, and validation RMSE. The loop only repeats the fit/predict pattern you practiced.

In [ ]:
# [PROVIDED EXPERIMENT] Repeat the fit/predict pattern for three settings.
# Each row uses the same training and validation observations. No test data.
# Keep these lists and the initial task-cell settings fixed for required work.
# These loops perform the required three-value parameter comparison for you.
ridge_alphas = [0.1, 1.0, 10.0]
svr_cs = [10.0, 100.0, 1000.0]
# This is True only when both student prediction placeholders have been completed.
tuning_ready = ridge_valid_pred is not None and svr_valid_pred is not None
if tuning_ready:
    # Start an empty results table; append one row after each fit.
    ridge_rows = []
    # The indented lines repeat once for each value in ridge_alphas.
    for alpha in ridge_alphas:
        candidate = Ridge(alpha=alpha)
        candidate.fit(X_train_scaled, y_train)
        # Training predictions measure fit to data the model has already seen.
        train_prediction = candidate.predict(X_train_scaled)
        # Validation predictions measure performance on the next, unseen cuts.
        valid_prediction = candidate.predict(X_valid_scaled)
        ridge_rows.append([alpha,
            np.sqrt(mean_squared_error(y_train, train_prediction)),
            np.sqrt(mean_squared_error(y_valid, valid_prediction))])
    ridge_tuning = pd.DataFrame(ridge_rows,
        columns=['alpha', 'Train RMSE (µm)', 'Validation RMSE (µm)'])

### Experiment B — repeat SVR for three C values

**Run unchanged.** This repeats the same process with C = 10, 100, 1000. All candidates receive the same scaled inputs. Epsilon and the gamma rule stay fixed.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if tuning_ready:
    # Store SVR results separately so alpha and C are not mixed.
    svr_rows = []
    # A fresh model is created for every C; fits do not accumulate.
    for c_value in svr_cs:
        candidate = SVR(kernel='rbf', C=c_value, epsilon=1.0, gamma='scale')
        candidate.fit(X_train_scaled, y_train)
        # Training predictions measure fit to data the model has already seen.
        train_prediction = candidate.predict(X_train_scaled)
        # Validation predictions measure performance on the next, unseen cuts.
        valid_prediction = candidate.predict(X_valid_scaled)
        svr_rows.append([c_value,
            np.sqrt(mean_squared_error(y_train, train_prediction)),
            np.sqrt(mean_squared_error(y_valid, valid_prediction))])
    svr_tuning = pd.DataFrame(svr_rows,
        columns=['C', 'Train RMSE (µm)', 'Validation RMSE (µm)'])

### Read the tables and curves

**Run unchanged; interpret the output.** Blue is training error; orange is validation error. Lower is better. Moving right means a tenfold parameter increase. Check whether the two errors move together before adding your observation.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if tuning_ready:
    display(ridge_tuning.round(3))
    display(svr_tuning.round(3))

    # Log axes place tenfold parameter increases at equal distances.
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
    axes[0].plot(ridge_alphas, ridge_tuning['Train RMSE (µm)'], 'o-', label='Train')
    axes[0].plot(ridge_alphas, ridge_tuning['Validation RMSE (µm)'], 's-', label='Validation')
    axes[1].plot(svr_cs, svr_tuning['Train RMSE (µm)'], 'o-', label='Train')
    axes[1].plot(svr_cs, svr_tuning['Validation RMSE (µm)'], 's-', label='Validation')
    axes[0].set(title='Ridge', xlabel='alpha (log scale)', ylabel='RMSE (µm)')
    axes[1].set(title='RBF SVR', xlabel='C (log scale)')
    for ax in axes:
        ax.set_xscale('log')
        ax.set_ylim(bottom=0)
        ax.grid(alpha=0.2)
        ax.legend()
    fig.tight_layout()
    plt.show()

### Keep the setting with the smallest validation RMSE

**Run unchanged.** `idxmin()` finds the row with the lowest validation error; `.loc[row, column]` reads the setting from that row. The code rebuilds Ridge and SVR with these selected settings so the next comparison uses them. Training error and test data do not choose the setting.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if tuning_ready:
    # Select settings ONLY from validation RMSE; ties keep the first value.
    # Find the smallest validation RMSE row, read its alpha, and convert it to a Python number.
    best_alpha = float(ridge_tuning.loc[ridge_tuning['Validation RMSE (µm)'].idxmin(), 'alpha'])
    # Apply the same validation-only rule to the SVR table to obtain C.
    best_c = float(svr_tuning.loc[svr_tuning['Validation RMSE (µm)'].idxmin(), 'C'])
    ridge_model = Ridge(alpha=best_alpha)
    ridge_model.fit(X_train_scaled, y_train)
    ridge_valid_pred = ridge_model.predict(X_valid_scaled)
    svr_model = SVR(kernel='rbf', C=best_c, epsilon=1.0, gamma='scale')
    svr_model.fit(X_train_scaled, y_train)
    svr_valid_pred = svr_model.predict(X_valid_scaled)
    print('Validation-selected alpha / C:', best_alpha, best_c)
else:
    print('Complete the four fit/predict edits, then restart and Run all.')
if tuning_ready:
    # Compare your preserved initial fits with the validation-selected settings.
    initial_vs_tuned = pd.DataFrame({
        'Model': ['Ridge', 'SVR'],
        'Initial setting': ['alpha=' + str(ridge_initial_model.alpha), 'C=' + str(svr_initial_model.C)],
        'Initial validation RMSE (µm)': [ridge_initial_rmse, svr_initial_rmse],
        'Selected setting': ['alpha=' + str(best_alpha), 'C=' + str(best_c)],
        'Selected validation RMSE (µm)': [
            np.sqrt(mean_squared_error(y_valid, ridge_valid_pred)),
            np.sqrt(mean_squared_error(y_valid, svr_valid_pred))],
    })
    display(initial_vs_tuned.round(3))


## Required Checkpoint 3 — Compare and select using validation

Run the supplied comparison. Each row is one model family; Ridge and SVR use the settings selected above.
These are the best among the supplied settings on this split, not global optima. **Choose the lowest pooled validation RMSE**.
Use MAE, R² and cutter-specific RMSE to explain strengths/limits, but do not
change the selection rule after seeing results. Training RMSE describes fit
to known data; it is not the selection score. A train–validation gap alone
does not identify its cause.

The supplied reporting code and loops only assemble results. You do not
need to recreate them. They evaluate all candidates on identical observations.

### Build the four-model comparison table

**Reporting code — run unchanged.** Lists keep models and predictions in matching order. The outer loop visits each model; the inner loop reports each cutter separately. A `mask` is a sequence of True/False values that selects only that cutter. You are assessed on reading the results, not writing these loops.

In [ ]:
# [PROVIDED REPORTING — RUN UNCHANGED] The guard prevents errors in an unfinished starter.
all_models_ready = (ridge_valid_pred is not None and polynomial_valid_pred is not None
                    and svr_valid_pred is not None)
selected_name = None
if all_models_ready:
    # Each list below follows the same model order.
    names = ['Linear', 'Ridge', 'Polynomial', 'SVR']
    validation_predictions = [linear_valid_pred, ridge_valid_pred, polynomial_valid_pred, svr_valid_pred]
    training_predictions = [linear_train_pred, ridge_model.predict(X_train_scaled),
                            polynomial_model.predict(X_train_poly), svr_model.predict(X_train_scaled)]
    rows = []
    cutter_rows = []
    for index in range(4):
        prediction = validation_predictions[index]
        rows.append([names[index], np.sqrt(mean_squared_error(y_train, training_predictions[index])),
                     mean_absolute_error(y_valid, prediction), np.sqrt(mean_squared_error(y_valid, prediction)),
                     r2_score(y_valid, prediction)])
        for cutter in ['c1', 'c4', 'c6']:
            mask = valid['cutter_id'] == cutter
            cutter_rows.append([names[index], cutter, np.sqrt(mean_squared_error(y_valid[mask], prediction[mask]))])
    comparison = pd.DataFrame(rows, columns=['Model', 'Train RMSE (µm)', 'Validation MAE (µm)',
                                           'Validation RMSE (µm)', 'Validation R²'])
    by_cutter = pd.DataFrame(cutter_rows, columns=['Model', 'Cutter', 'Validation RMSE (µm)'])

### Read the model choice

**Run unchanged.** The first table compares models overall. The second compares cutters. Select the smallest overall validation RMSE, then use the other values to explain limitations in Checkpoint 3.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    display(comparison.round(3))
    display(by_cutter.pivot(index='Model', columns='Cutter', values='Validation RMSE (µm)').round(2))
    # idxmin returns the row with minimum RMSE; equal values keep the first candidate.
    # idxmin returns a ROW LABEL, not the RMSE itself; equal minima keep the first row.
    selected_index = comparison['Validation RMSE (µm)'].idxmin()
    # Read the Model column at that winning row so the final stage knows which model to create.
    selected_name = comparison.loc[selected_index, 'Model']
    print('Selected using validation only:', selected_name)
else:
    print('Complete the four CP2 edits, then run all before selecting a model.')

### Read the validation comparison

Lower validation RMSE is better. Compare the validation-selected Ridge
and SVR against Linear and fixed degree-2 Polynomial. Training RMSE is
shown for context; never choose solely from training fit. Negative wear
predictions, if any, remain in all metrics; no clipping improves a score.

In [ ]:
# [PROVIDED PLOT] Each family uses the same validation observations.
if all_models_ready:
    fig, ax = plt.subplots(figsize=(9, 3.8))
    # Four x positions, one for each model; offsets separate the two bars.
    positions = np.arange(4)
    ax.bar(positions - 0.18, comparison['Train RMSE (µm)'], 0.36, label='Train')
    ax.bar(positions + 0.18, comparison['Validation RMSE (µm)'], 0.36, label='Validation')
    ax.set_xticks(positions, names)
    ax.set(ylabel='RMSE (µm)', title='Model comparison after validation-based parameter selection')
    ax.legend()
    fig.tight_layout()
    plt.show()

### Checkpoint 3 response — complete before running the final test

Name the selected candidate and its chosen setting and cite its validation RMSE and one competitor's
RMSE. Use the cutter table or train–validation gap to state one caution. Explain
why you did not choose the candidate from test results.

**Your response:** TODO: Write 2–3 sentences.

## Required Checkpoint 4 — Refit, then evaluate the reserved test

Our model choice and settings are now fixed. We can use the original train
**and validation** rows to learn a final model: 220 cuts per cutter, 660 total.
This does not undo validation: validation already served its selection role.

The code creates a **fresh scaler and fresh model**, fits them on these 660
rows, and transforms the test inputs without fitting to them. A polynomial
choice also receives a fresh polynomial transformer. Test labels are used
only after prediction to calculate errors. Do not switch models based on test.
The conditionals are supplied implementation; run them unchanged.

### Final evaluation A — prepare the first 70% for refitting

**Run after writing Checkpoint 3.** `refit` stacks the original training and validation rows. A new scaler learns from those 660 rows. It only transforms the reserved test inputs; it never learns their mean or spread.

In [ ]:
# [PROVIDED FINAL EVALUATION] Execute only after recording the CP3 response.
if all_models_ready:
    # Stack training and validation rows after selection is locked: 471 + 189 = 660 cuts.
    refit = pd.concat([train, valid], ignore_index=True)
    # Extract the same 11 sensor inputs from the combined refit table.
    X_refit = refit[default_features]
    # Use the matching 660 measured wear values to learn the final model.
    y_refit = refit['wear_mean_um']
    # Extract only sensor inputs from the 285 reserved cuts.
    X_test = test[default_features]
    # Keep the measured test values separate; they are used to score predictions, not fit.
    y_test = test['wear_mean_um']
    # Create a fresh scaler rather than reusing statistics learned from only the first 50%.
    final_scaler = StandardScaler()
    # Learn means/spreads from the 660 refit rows and standardize those same rows.
    X_refit_scaled = final_scaler.fit_transform(X_refit)
    # Apply the refit statistics unchanged; do not fit a scaler on test data.
    X_test_scaled = final_scaler.transform(X_test)

### Final evaluation B — recreate the selected model

**Run unchanged.** Only one `if/elif/else` branch runs, based on the chosen model name. It keeps the validation-selected parameter. Polynomial expansion is applied only if Polynomial won; otherwise the 11 scaled columns are used.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    # Use the selected family with the validation-selected settings.
    if selected_name == 'Linear':
        final_model = LinearRegression()
    elif selected_name == 'Ridge':
        final_model = Ridge(alpha=best_alpha)
    elif selected_name == 'Polynomial':
        final_model = LinearRegression()
    else:
        final_model = SVR(kernel='rbf', C=best_c, epsilon=1.0, gamma='scale')

    # Default inputs have 11 columns; only the Polynomial branch changes this.
    X_refit_final = X_refit_scaled
    X_test_final = X_test_scaled
    if selected_name == 'Polynomial':
        final_polynomial = PolynomialFeatures(degree=2, include_bias=False)
        X_refit_final = final_polynomial.fit_transform(X_refit_scaled)
        X_test_final = final_polynomial.transform(X_test_scaled)

### Final evaluation C — predict, then calculate errors

**Run unchanged.** Fit the fresh model on 660 rows, predict the 285 reserved cuts, and only then compare predictions with measured test wear. The cutter table helps locate errors hidden by the overall average. Do not retune from this result.

In [ ]:
# [PROVIDED — RUN UNCHANGED]
if all_models_ready:
    # Learn the selected model from refit inputs and targets; the test set is still excluded.
    final_model.fit(X_refit_final, y_refit)
    # Predict one wear value for each reserved cut without supplying measured test wear.
    test_pred = final_model.predict(X_test_final)
    # Convert the measured series to an array and subtract corresponding predictions.
    # Positive residual = underprediction; values remain in micrometers.
    test_residual = y_test.to_numpy() - test_pred
    # MAE averages absolute errors; RMSE gives larger errors more weight.
    final_mae = mean_absolute_error(y_test, test_pred)
    # mean_squared_error averages squared residuals; sqrt returns the result to µm.
    final_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
    # R² is unitless and can be negative; it compares with the evaluated target mean.
    final_r2 = r2_score(y_test, test_pred)
    print('Selected model:', selected_name, '| Refit / test rows:', len(refit), len(test))
    print('Test MAE (µm):', round(final_mae, 2))
    print('Test RMSE (µm):', round(final_rmse, 2))
    print('Test R² (unitless):', round(final_r2, 3))
    test_rows = []
    for cutter in ['c1', 'c4', 'c6']:
        mask = test['cutter_id'] == cutter
        test_rows.append([cutter, mean_absolute_error(y_test[mask], test_pred[mask]),
                          np.sqrt(mean_squared_error(y_test[mask], test_pred[mask]))])
    test_by_cutter = pd.DataFrame(test_rows, columns=['Cutter', 'Test MAE (µm)', 'Test RMSE (µm)'])
    display(test_by_cutter.round(2))
else:
    print('Final test waits until all four validation candidates are available.')

### Read the final plots

**Run unchanged.** Each column is one cutter. Top: measured and predicted wear. Bottom: measured minus predicted. Positive residuals mean underprediction; negative residuals mean overprediction. The dashed zero line means exact agreement.

In [ ]:
# [PROVIDED PLOT] Separate cutters so disconnected trajectories are not joined.
if all_models_ready:
    fig, axes = plt.subplots(2, 3, figsize=(12, 6), sharex=True, sharey='row')
    for index, cutter in enumerate(['c1', 'c4', 'c6']):
        mask = test['cutter_id'] == cutter
        cuts = test.loc[mask, 'cut_number']
        axes[0, index].plot(cuts, y_test[mask], label='Measured', color='#273B43')
        axes[0, index].plot(cuts, test_pred[mask], label='Predicted', color='#C08923')
        axes[0, index].set(title=cutter, ylabel='Mean flute wear (µm)')
        axes[1, index].plot(cuts, test_residual[mask], color='#2468A0')
        axes[1, index].axhline(0, color='black', linestyle='--')
        axes[1, index].set(xlabel='Cut number', ylabel='Residual (µm)')
    axes[0, 0].legend(fontsize=9)
    fig.suptitle('Selected model on later cuts: ' + selected_name)
    fig.tight_layout()
    plt.show()

### Best candidate does not mean sufficient accuracy

The selected model has the lowest validation RMSE among the supplied candidates
on this split. It can still make large errors on later cuts. A negative test R²
means it has more squared error than the test-set-mean reference; that reference
is not a prediction available before seeing the test targets.

**A poor score alone does not show that your code is wrong.** First check that
your data and workflow match the instructions, then report what the results
support. There is no target RMSE that you must reach by changing the experiment.
Explain the errors in µm and their engineering implications; do not tune on test
data to obtain a more attractive result. Suitability for a real application would
also require an application-specific acceptable error and further validation.


### Checkpoint 4 response

Interpret the selected model's test MAE in µm and describe one cutter-specific
residual pattern or error difference. Explain why this experiment does not
demonstrate performance on an unseen cutter or prove that a sensor causes wear.

**Your response:** TODO: Write 2–3 sentences.

## Optional — inspect the selected model across all c6 cuts

**Ungraded; run unchanged after the final evaluation.** Use the final selected
model to predict wear for all 315 cuts of c6. Do not fit anything again.

The first 220 cuts were used to refit the final model; agreement there is
in-sample fit, not independent validation. Only cuts 221–315 remain the test
segment. Background shading separates these two roles. The original validation
segment is now part of refitting, so it is no longer held out from this final model.

Compare the measured and predicted curves. Where does the gap grow, and does
the model overpredict or underpredict? Could a good fit in the first region alone
justify using this model on later cuts? No written submission is required.
Do not choose new settings from this plot or report an all-cuts score as test performance.


In [ ]:
# [OPTIONAL — PROVIDED] Reuse the final model; no fitting or tuning here.
if all_models_ready:
    # c6 is already ordered by cut number. Keep the same 11 sensor columns.
    X_c6_all = c6[default_features]
    # Apply the scaler learned from the pooled first 220 cuts of all three cutters.
    X_c6_scaled = final_scaler.transform(X_c6_all)
    X_c6_final = X_c6_scaled
    # Reuse the fitted expansion only if Polynomial was selected.
    if selected_name == 'Polynomial':
        X_c6_final = final_polynomial.transform(X_c6_scaled)
    # One prediction per cut, in µm; measured wear is not passed to predict.
    c6_all_pred = final_model.predict(X_c6_final)
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.axvspan(0.5, 220.5, color='#E7E7E7', label='Used in final refit (1–220)')
    ax.axvspan(220.5, 315.5, color='#E1F0E8', label='Reserved test (221–315)')
    ax.plot(c6['cut_number'], c6['wear_mean_um'], color='#263238', linewidth=2.2, label='Measured wear')
    ax.plot(c6['cut_number'], c6_all_pred, color='#D55E00', linewidth=1.8, label='Predicted wear')
    ax.axvline(220.5, color='#555555', linestyle='--', linewidth=1)
    ax.set(xlabel='Cut number', ylabel='Mean flute wear (µm)', xlim=(1, 315),
           title='c6: selected model across all cuts — ' + selected_name)
    ax.legend(loc='upper left', fontsize=10)
    fig.tight_layout()
    plt.show()
else:
    print('Complete the required model cells and final evaluation before this optional plot.')


## Optional challenge — interpretation only, ungraded

Using the existing tables, identify a model with a noticeable train–validation
gap. Give two possible explanations without claiming that this gap proves
one specific cause. No extra code, fitting or parameter search is required.

## What this evaluation does—and what comes later

We tested **later cuts of cutters already represented in training**.
Even the same cutter may later produce sensor values unlike its early values.
ID/OOD describes relationships between evaluation and training conditions;
being the same cutter alone does not guarantee similar input distributions.
Formal distribution-shift and unseen-cutter evaluations belong to Week 6.

Here the sensor summaries from a cut estimate its associated observed wear.
This is not prediction of future wear before observing that cut, and it is
not remaining-useful-life forecasting. A strong score is not evidence of causation.

## Submission and reproducibility checklist

Restart the runtime/kernel and select **Run all** after finishing your edits.
Check that all four candidates appear, a selected model is printed, and final
test results are visible. Fix code errors before exporting.

Rendered notebook checkboxes are not clickable. To record completion, edit
this Markdown cell and change `[ ]` to `[x]`.

- [ ] I entered my name and WSU AccessID.
- [ ] I completed four CP2 fit/predict edits and the four checkpoint responses.
- [ ] I explained the validation-based choice before interpreting test results.
- [ ] I restarted, ran all cells and checked required outputs and labels/units.
- [ ] I saved `Lab04_Firstname_Lastname.ipynb` and a matching PDF.
- [ ] I checked that figures and written responses are readable in the PDF.
- [ ] I submitted both files through Canvas. No separate report is required.

PDF export issues receive corrective comments, not point deductions. Required
work and submission files still need to be provided; see Canvas for active instructions.

## References

- [PHM Society: 2010 Data Challenge](https://phmsociety.org/phm_competition/2010-phm-society-conference-data-challenge/).
- Course-derived feature table and provenance: [README](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/README.md),
  [data dictionary](https://github.com/WSU-AI-in-ME/ai-in-me-1/blob/main/data/phm2010/features/DATA_DICTIONARY.md).
- scikit-learn documentation: [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html),
  [Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html),
  [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html),
  [SVR](https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVR.html),
  [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html),
  [regression metrics](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics).

**Dataset acknowledgment:** PHM Society 2010 Data Challenge; course-derived
scalar summaries and mean-flute-wear target. Cite the original challenge and
course feature documentation when reusing this table.